# Initialiazation Packages

In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from dataclasses import dataclass

# -------------------------------------------------------------------
# Units:
# Use energies/frequencies in ANGULAR units.
# Convenient choice: time in ns, energies in rad/ns.
# Then 1 GHz corresponds to 2*pi rad/ns.
# -------------------------------------------------------------------
GHz = 2.0 * np.pi  # multiply a value in GHz by this


## Fluxonium Lyapunov Calculation

In [67]:
import numpy as np
from dataclasses import dataclass
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, root


# ============================================================
# Parameters
# ============================================================

@dataclass
class FluxoniumParams:
    EC: float
    EJ: float
    EL: float
    phi_ext0: float
    omega_d: float
    A_charge: float = 0.0   # charge drive amplitude
    A_flux: float = 0.0     # flux drive amplitude


# ============================================================
# Classical driven fluxonium
#
# H(phi,n,t) =
#   4 EC n^2
#   + EL/2 * [phi - phi_ext0 - A_flux cos(omega_d t)]^2
#   - EJ cos(phi)
#   + A_charge cos(omega_d t) * n
#
# Equations:
#   dot(phi) = 8 EC n + A_charge cos(omega_d t)
#   dot(n)   = -EL [phi - phi_ext0 - A_flux cos(omega_d t)] - EJ sin(phi)
# ============================================================

def fluxonium_rhs(t, u, p: FluxoniumParams):
    phi, n = u

    dphi = 8.0 * p.EC * n + p.A_charge * np.cos(p.omega_d * t)
    dn = -p.EL * (phi - p.phi_ext0 - p.A_flux * np.cos(p.omega_d * t)) - p.EJ * np.sin(phi)

    return np.array([dphi, dn], dtype=float)


def fluxonium_jacobian(u, p: FluxoniumParams, t):
    """
    Jacobian with respect to the state variables (phi, n).
    Note: it depends on phi, but not explicitly on t except through the trajectory.
    """
    phi, n = u
    K = p.EL + p.EJ * np.cos(phi)

    J = np.array([
        [0.0,        8.0 * p.EC],
        [-K,         0.0]
    ], dtype=float)

    return J


# ============================================================
# State + tangent evolution
#
# Y = [phi, n, G11, G12, G21, G22]
# where G is 2x2.
#
# dG/dt = J(u(t), t) G
# ============================================================

def fluxonium_state_tangent_rhs(t, Y, p: FluxoniumParams):
    phi, n = Y[0], Y[1]

    # Base dynamics
    dphi = 8.0 * p.EC * n + p.A_charge * np.cos(p.omega_d * t)
    dn = -p.EL * (phi - p.phi_ext0 - p.A_flux * np.cos(p.omega_d * t)) - p.EJ * np.sin(phi)

    # Tangent matrix
    G = Y[2:].reshape((2, 2))
    J = fluxonium_jacobian([phi, n], p, t)
    dG = J @ G

    dY = np.empty_like(Y)
    dY[0] = dphi
    dY[1] = dn
    dY[2:] = dG.ravel()

    return dY


# ============================================================
# Lyapunov spectrum
# ============================================================

def lyapunov_spectrum_fluxonium(
    u0,
    p: FluxoniumParams,
    T_trans=100.0,
    T_total=2000.0,
    Delta_r=None,
    rtol=1e-10,
    atol=1e-12,
    method="DOP853",
):
    """
    Compute the full 2D Lyapunov spectrum for driven fluxonium.

    Parameters
    ----------
    u0 : array-like, shape (2,)
        Initial condition [phi0, n0].
    p : FluxoniumParams
        Model parameters.
    T_trans : float
        Transient time discarded from exponent accumulation.
    T_total : float
        Time used for actual accumulation after transient.
    Delta_r : float or None
        Reorthonormalization interval.
        If None, use one drive period.
    """

    u0 = np.asarray(u0, dtype=float)

    if Delta_r is None:
        Delta_r = 2.0 * np.pi / p.omega_d

    # Initial combined state
    Y = np.zeros(6, dtype=float)
    Y[0:2] = u0
    Y[2:] = np.eye(2).ravel()

    t = 0.0
    t_end = T_trans + T_total
    nsteps = int(np.ceil(t_end / Delta_r))

    lam_sum = np.zeros(2, dtype=float)
    elapsed = 0.0

    times = []
    lam_max_t = []

    for k in range(nsteps):
        sol = solve_ivp(
            fun=lambda tt, yy: fluxonium_state_tangent_rhs(tt, yy, p),
            t_span=(t, t + Delta_r),
            y0=Y,
            method=method,
            rtol=rtol,
            atol=atol,
            dense_output=False,
        )

        if not sol.success:
            raise RuntimeError(f"Integration failed at step {k}: {sol.message}")

        Y = sol.y[:, -1].copy()
        t = sol.t[-1]

        G = Y[2:].reshape((2, 2))

        # QR reorthonormalization
        Q, R = np.linalg.qr(G)

        # Make diagonal of R positive when possible
        for j in range(2):
            if R[j, j] < 0:
                R[j, :] *= -1.0
                Q[:, j] *= -1.0

        Y[2:] = Q.ravel()

        if t > T_trans:
            lam_sum += np.log(np.abs(np.diag(R)))
            elapsed += Delta_r
            times.append(t)
            lam_max_t.append(np.max(lam_sum) / elapsed)

    lam = lam_sum / elapsed
    lam_max = np.max(lam)

    return lam, lam_max, np.array(times), np.array(lam_max_t)


# ============================================================
# Undriven fixed points
#
# For A_charge = A_flux = 0:
#   n* = 0
#   EL (phi* - phi_ext0) + EJ sin(phi*) = 0
# ============================================================

def undriven_fixed_point_equation(phi, p: FluxoniumParams):
    return p.EL * (phi - p.phi_ext0) + p.EJ * np.sin(phi)


def find_undriven_fixed_points(p: FluxoniumParams, phi_min=-6*np.pi, phi_max=6*np.pi, nscan=20000):
    """
    Find fixed points of the undriven system by scanning for sign changes.
    """
    if abs(p.A_charge) > 0 or abs(p.A_flux) > 0:
        raise ValueError("Set A_charge = A_flux = 0 to find true undriven fixed points.")

    grid = np.linspace(phi_min, phi_max, nscan)
    vals = undriven_fixed_point_equation(grid, p)

    roots = []

    for i in range(len(grid) - 1):
        a, b = grid[i], grid[i+1]
        fa, fb = vals[i], vals[i+1]

        if fa == 0.0:
            roots.append(a)
        elif fa * fb < 0:
            r = brentq(lambda x: undriven_fixed_point_equation(x, p), a, b)
            roots.append(r)

    # Remove duplicates
    roots = np.array(sorted(roots))
    if len(roots) == 0:
        return []

    unique_roots = [roots[0]]
    for r in roots[1:]:
        if abs(r - unique_roots[-1]) > 1e-8:
            unique_roots.append(r)

    return [{"phi": r, "n": 0.0} for r in unique_roots]


def classify_undriven_fixed_point(phi_star, p: FluxoniumParams, tol=1e-12):
    """
    Linear stability of undriven fixed point.
    J* = [[0, 8EC], [-(EL + EJ cos(phi*)), 0]]
    """
    K = p.EL + p.EJ * np.cos(phi_star)

    if K > tol:
        omega_loc = np.sqrt(8.0 * p.EC * K)
        eigs = np.array([1j * omega_loc, -1j * omega_loc])
        return {
            "type": "elliptic",
            "K": K,
            "eigenvalues": eigs,
            "omega_local": omega_loc
        }

    elif K < -tol:
        lam = np.sqrt(8.0 * p.EC * (-K))
        eigs = np.array([lam, -lam])
        return {
            "type": "hyperbolic",
            "K": K,
            "eigenvalues": eigs,
            "growth_rate": lam
        }

    else:
        return {
            "type": "parabolic",
            "K": K,
            "eigenvalues": np.array([0.0, 0.0])
        }


# ============================================================
# Stroboscopic map (for the driven system)
#
# For a periodically driven system, the relevant "fixed points"
# are fixed points of the Poincare map, not of the instantaneous flow.
# ============================================================

def stroboscopic_map(
    u0,
    p: FluxoniumParams,
    nperiods=1,
    phase_fraction=0.0,
    rtol=1e-11,
    atol=1e-13,
    method="DOP853",
):
    """
    Map u0 -> u after nperiods of the drive, starting at time phase_fraction*T.
    """
    u0 = np.asarray(u0, dtype=float)
    T = 2.0 * np.pi / p.omega_d
    t0 = phase_fraction * T
    tf = t0 + nperiods * T

    sol = solve_ivp(
        fun=lambda tt, uu: fluxonium_rhs(tt, uu, p),
        t_span=(t0, tf),
        y0=u0,
        method=method,
        rtol=rtol,
        atol=atol,
        dense_output=False,
    )

    if not sol.success:
        raise RuntimeError(sol.message)

    return sol.y[:, -1].copy()


def poincare_residual(u, p: FluxoniumParams, nperiods=1, phase_fraction=0.0):
    u = np.asarray(u, dtype=float)
    uT = stroboscopic_map(u, p, nperiods=nperiods, phase_fraction=phase_fraction)
    return uT - u


def find_periodic_point(
    u_guess,
    p: FluxoniumParams,
    nperiods=1,
    phase_fraction=0.0,
):
    """
    Find a period-nperiods point of the stroboscopic map.
    """
    u_guess = np.asarray(u_guess, dtype=float)

    sol = root(
        fun=lambda u: poincare_residual(u, p, nperiods=nperiods, phase_fraction=phase_fraction),
        x0=u_guess,
    )

    return sol.x, sol


# ============================================================
# Monodromy matrix and stability of periodic orbits
# ============================================================

def monodromy_matrix(
    u_star,
    p: FluxoniumParams,
    nperiods=1,
    phase_fraction=0.0,
    rtol=1e-11,
    atol=1e-13,
    method="DOP853",
):
    """
    Tangent map over nperiods of the drive, starting from periodic point u_star.
    """
    u_star = np.asarray(u_star, dtype=float)

    T = 2.0 * np.pi / p.omega_d
    t0 = phase_fraction * T
    tf = t0 + nperiods * T

    Y0 = np.zeros(6, dtype=float)
    Y0[0:2] = u_star
    Y0[2:] = np.eye(2).ravel()

    sol = solve_ivp(
        fun=lambda tt, YY: fluxonium_state_tangent_rhs(tt, YY, p),
        t_span=(t0, tf),
        y0=Y0,
        method=method,
        rtol=rtol,
        atol=atol,
        dense_output=False,
    )

    if not sol.success:
        raise RuntimeError(sol.message)

    Yf = sol.y[:, -1]
    M = Yf[2:].reshape((2, 2)).copy()

    return M


def classify_periodic_orbit(
    u_star,
    p: FluxoniumParams,
    nperiods=1,
    phase_fraction=0.0,
    tol=1e-8,
):
    """
    Classify periodic point through Floquet multipliers of the Poincare map.
    """
    M = monodromy_matrix(u_star, p, nperiods=nperiods, phase_fraction=phase_fraction)
    mu = np.linalg.eigvals(M)

    # For a 2D Hamiltonian stroboscopic map, det(M) should be ~ 1
    detM = np.linalg.det(M)
    trM = np.trace(M)

    if np.all(np.isreal(mu)) and np.max(np.abs(mu)) > 1.0 + tol:
        orbit_type = "hyperbolic"
    elif np.all(np.abs(np.abs(mu) - 1.0) < 1e-4):
        orbit_type = "elliptic_or_neutral"
    else:
        orbit_type = "near_resonant_or_unclear"

    return {
        "type": orbit_type,
        "M": M,
        "multipliers": mu,
        "detM": detM,
        "trM": trM,
    }



In [68]:

# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    # --------------------------------------------------------
    # Example 1: driven case, Lyapunov exponent
    # --------------------------------------------------------
    p = FluxoniumParams(
        EC=1.0,
        EJ=8.5,
        EL=0.5,
        phi_ext0=np.pi,
        omega_d=2.0,
        A_charge=0.3,
        A_flux=0.0,
    )

    u0 = np.array([np.pi + 0.2, 0.0])

    lam, lam_max, times, lam_max_t = lyapunov_spectrum_fluxonium(
        u0=u0,
        p=p,
        T_trans=200.0,
        T_total=4000.0,
        Delta_r=2.0 * np.pi / p.omega_d,
    )

    print("Lyapunov spectrum:", lam)
    print("Largest Lyapunov exponent:", lam_max)

    # --------------------------------------------------------
    # Example 2: undriven fixed points
    # --------------------------------------------------------
    p0 = FluxoniumParams(
        EC=1.0,
        EJ=8.5,
        EL=0.5,
        phi_ext0=np.pi,
        omega_d=2.0,
        A_charge=0.0,
        A_flux=0.0,
    )

    fps = find_undriven_fixed_points(p0)

    print("\nUndriven fixed points:")
    for fp in fps:
        stab = classify_undriven_fixed_point(fp["phi"], p0)
        print(fp, stab)

    # --------------------------------------------------------
    # Example 3: driven periodic point of the stroboscopic map
    # --------------------------------------------------------
    u_guess = np.array([np.pi + 1.0, 0.0])
    u_star, info = find_periodic_point(u_guess, p, nperiods=1, phase_fraction=0.0)

    print("\nDriven period-1 point:", u_star)
    print("Root success:", info.success, info.message)

    orbit_info = classify_periodic_orbit(u_star, p, nperiods=1, phase_fraction=0.0)
    print("Periodic orbit stability:", orbit_info)

Lyapunov spectrum: [ 0.15480722 -0.15480722]
Largest Lyapunov exponent: 0.15480722282609513

Undriven fixed points:
{'phi': np.float64(-11.525631951662564), 'n': 0.0} {'type': 'elliptic', 'K': np.float64(4.79745629352643), 'eigenvalues': array([0.+6.19513118j, 0.-6.19513118j]), 'omega_local': np.float64(6.195131180871915)}
{'phi': np.float64(-10.340611930146759), 'n': 0.0} {'type': 'hyperbolic', 'K': np.float64(-4.677599819469278), 'eigenvalues': array([ 6.11725417, -6.11725417]), 'growth_rate': np.float64(6.117254167986992)}
{'phi': np.float64(-5.733889030808105), 'n': 0.0} {'type': 'elliptic', 'K': np.float64(7.749583175430326), 'eigenvalues': array([0.+7.87379612j, 0.-7.87379612j]), 'omega_local': np.float64(7.873796124071451)}
{'phi': np.float64(-3.545899854130123), 'n': 0.0} {'type': 'hyperbolic', 'K': np.float64(-7.314688796734002), 'eigenvalues': array([ 7.64967387, -7.64967387]), 'growth_rate': np.float64(7.649673873693702)}
{'phi': np.float64(0.17538074382681434), 'n': 0.0} {'